In [1]:
#Given a mutual fund’s historical risk, returns, cost, and consistency metrics,how likely is it to perform well in the future relative to peers?
#ANS: We built a goal-based mutual fund recommendation system with return projection and risk alignment

"""
Suitability Score =
  w1 * Composite Score
+ w2 * Sharpe Ratio
+ w3 * Consistency Score
- w4 * Expense Ratio
- w5 * Risk Penalty
"""

'\nSuitability Score =\n  w1 * Composite Score\n+ w2 * Sharpe Ratio\n+ w3 * Consistency Score\n- w4 * Expense Ratio\n- w5 * Risk Penalty\n'

In [2]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_excel("../Excel/mutual_funds.xlsx")

In [4]:
df.head()

,scheme_name,min_sip,min_lumpsum,expense_ratio,fund_size_cr,fund_age_yr,fund_manager,sortino_ratio,alpha,standard_deviation,...,sub_category,returns_1yr,returns_3yr,returns_5yr,risk-adjusted return score,cost efficiency score,consistency score,fund stability,composite_score,rank
0,Quant Infrastructure Fund - Regular Plan IDCW,1000,5000,0.586682,888.937047,10,Vasav Sahgal,3.726774,27.810968,24.259553,...,Sectoral / Thematic Mutual Funds,3.958764,63.090821,22.539075,2.600659,1.704501,29.862887,36.642763,0.704149,1.0
1,Quant Infrastructure Fund - Direct Plan Growth,1000,5000,0.629340,851.092153,11,Vasav Sahgal,3.408510,28.792777,24.292619,...,Sectoral / Thematic Mutual Funds,8.376421,60.112592,22.557321,2.474521,1.588966,30.348778,35.03501,0.701785,2.0
2,Quant Infrastructure Fund - Regular Plan Growth,1000,5000,0.577551,731.522839,10,Vasav Sahgal,3.767372,27.471415,25.389421,...,Sectoral / Thematic Mutual Funds,7.311768,62.288565,20.664012,2.453328,1.731449,30.088115,28.812112,0.697625,3.0
3,Quant Infrastructure Fund - Regular Plan IDCW,1000,5000,0.591545,768.126541,9,Vasav Sahgal,3.516540,26.538881,22.417272,...,Sectoral / Thematic Mutual Funds,5.956095,62.014558,22.404504,2.766374,1.690489,30.125053,34.264943,0.697512,4.0
4,Quant Infrastructure Fund - Regular Plan Growth,1000,5000,0.624836,879.865799,9,Vasav Sahgal,3.456080,28.775877,21.597198,...,Sectoral / Thematic Mutual Funds,8.193366,60.586547,21.827366,2.805297,1.600421,30.202426,40.739812,0.697401,5.0


In [5]:
df.shape

(3031, 27)

In [6]:
df.describe()

,expense_ratio,sortino_ratio,alpha,beta,sharpe,returns_1yr,returns_3yr,returns_5yr,composite_score,rank
count,3030.000000,3030.000000,3030.000000,3030.000000,3030.000000,3030.000000,3031.000000,3031.000000,3030.000000,3030.000000
mean,0.551772,4.073348,11.387331,0.869333,1.945333,7.283633,41.135391,16.308736,0.506755,1515.500000
std,0.245373,0.943230,5.057351,0.303870,0.196582,7.421805,10.713251,3.862844,0.064634,874.829983
min,0.072027,2.063522,3.386165,0.530000,1.370000,-18.982474,12.503774,7.993444,0.383299,1.000000
25%,0.342452,3.533179,7.767441,0.750000,1.880000,3.338745,35.730702,13.665744,0.459267,758.250000
50%,0.584860,3.972362,10.043348,0.840000,1.945000,7.268550,41.641648,16.003408,0.500248,1515.500000
75%,0.732491,4.446784,14.248807,0.920000,2.080000,10.906011,45.405224,19.448298,0.543252,2272.750000
max,1.099229,7.990798,29.227260,2.360000,2.300000,26.114468,73.316449,24.688237,0.704149,3030.000000


In [7]:
df.columns.tolist()

['scheme_name',
 'min_sip',
 'min_lumpsum',
 'expense_ratio',
 'fund_size_cr',
 'fund_age_yr',
 'fund_manager',
 'sortino_ratio',
 'alpha',
 'standard_deviation',
 'beta',
 'sharpe',
 'risk_level',
 'risk_bucket',
 'amc_name',
 'rating',
 'category',
 'sub_category',
 'returns_1yr',
 'returns_3yr',
 'returns_5yr',
 'risk-adjusted return score',
 'cost efficiency score',
 'consistency score',
 'fund stability',
 'composite_score',
 'rank']

In [8]:
df = df.rename(columns={
    "risk-adjusted return score": "risk_adjusted_return_score",
    "cost efficiency score": "cost_efficiency_score",
    "consistency score": "consistency_score",
    "fund stability": "fund_stability"
})

In [9]:
df.columns.tolist()

['scheme_name',
 'min_sip',
 'min_lumpsum',
 'expense_ratio',
 'fund_size_cr',
 'fund_age_yr',
 'fund_manager',
 'sortino_ratio',
 'alpha',
 'standard_deviation',
 'beta',
 'sharpe',
 'risk_level',
 'risk_bucket',
 'amc_name',
 'rating',
 'category',
 'sub_category',
 'returns_1yr',
 'returns_3yr',
 'returns_5yr',
 'risk_adjusted_return_score',
 'cost_efficiency_score',
 'consistency_score',
 'fund_stability',
 'composite_score',
 'rank']

In [11]:
print(df.columns.tolist())

['scheme_name', 'min_sip', 'min_lumpsum', 'expense_ratio', 'fund_size_cr', 'fund_age_yr', 'fund_manager', 'sortino_ratio', 'alpha', 'standard_deviation', 'beta', 'sharpe', 'risk_level', 'risk_bucket', 'amc_name', 'rating', 'category', 'sub_category', 'returns_1yr', 'returns_3yr', 'returns_5yr', 'risk_adjusted_return_score', 'cost_efficiency_score', 'consistency_score', 'fund_stability', 'composite_score', 'rank']


In [16]:
cols = ["returns_5yr", "sharpe", "standard_deviation"]

df[cols] = (
    df[cols]
    .replace("-", pd.NA)
    .apply(pd.to_numeric, errors="coerce")
)

In [17]:
df[cols] = df[cols].fillna(df[cols].median())

In [18]:
df["target_score_raw"] = (
    0.5 * df["returns_5yr"]
    + 0.3 * df["sharpe"]
    - 0.2 * df["standard_deviation"]
)

In [ ]:
num_features = [
    "standard_deviation",
    "sharpe",
    "sortino_ratio",
    "expense_ratio",
    "fund_size_cr",
    "consistency_score",
    "cost_efficiency_score",
    "fund_stability",
    "alpha",
    "beta"
]

cat_features = [
    "category",
    "risk_bucket"
]

In [ ]:
from sklearn.model_selection import train_test_split

X = df[num_features + cat_features]
y = df["target_score_raw"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

In [37]:
feature_cols = [
    "expense_ratio",
    "fund_size_cr",
    "fund_age_yr",
    "returns_1yr",
    "returns_3yr",
    "returns_5yr",
    "sharpe",
    "sortino_ratio",
    "alpha",
    "beta",
    "standard_deviation",
    "risk_adjusted_return_score",
    "cost_efficiency_score",
    "consistency_score",
    "fund_stability",
    "category",
    "risk_bucket"
]

X = df[feature_cols].copy()

# Ensure numeric columns are numeric
num_cols = [
    "expense_ratio", "fund_size_cr", "fund_age_yr",
    "returns_1yr", "returns_3yr", "returns_5yr",
    "sharpe", "sortino_ratio", "alpha", "beta",
    "standard_deviation", "risk_adjusted_return_score",
    "cost_efficiency_score", "consistency_score", "fund_stability"
]

X[num_cols] = X[num_cols].apply(pd.to_numeric, errors="coerce")


In [21]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer


In [22]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [23]:
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [24]:
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

In [25]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

In [26]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", model)
])

In [27]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['standard_deviation',
                                                   'sharpe', 'sortino_ratio',
                                                   'expense_ratio',
                                                   'fund_size_cr',
                                                   'consistency_score',
                                                   'cost_efficiency_score',
                                                   'fund_stability', 'alpha',
                                                   'beta']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['category',
                                                   'risk_bucket'])])),
                ('model',
                 RandomForestRegressor(n_estimators=200, random_state=42))])

In [38]:
df["predicted_raw_score"] = pipeline.predict(X)

df.sort_values("predicted_raw_score", ascending=False)[
    ["scheme_name", "predicted_raw_score", "target_score_raw"]
].head(5)


,scheme_name,predicted_raw_score,target_score_raw
281,Quant Multi Asset Fund - Direct Plan IDCW,8.810931,8.987936
459,Quant Multi Asset Fund - Regular Plan IDCW,8.698221,8.943774
301,Quant Multi Asset Fund - Regular Plan Growth,8.657363,8.854722
245,Quant Multi Asset Fund - Direct Plan IDCW,8.654700,8.937733
355,Quant Multi Asset Fund - Regular Plan IDCW,8.626516,8.767714


In [39]:
df[["returns_5yr", "sharpe", "standard_deviation", "target_score_raw"]].corr()

,returns_5yr,sharpe,standard_deviation,target_score_raw
returns_5yr,1.000000,-0.000173,0.475516,0.836059
sharpe,-0.000173,1.000000,0.110363,-0.034804
standard_deviation,0.475516,0.110363,1.000000,-0.084135
target_score_raw,0.836059,-0.034804,-0.084135,1.000000


with Z-score target

In [40]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df[["z_returns_5yr", "z_sharpe", "z_std_dev"]] = scaler.fit_transform(
    df[["returns_5yr", "sharpe", "standard_deviation"]]
)


In [41]:
df["target_score_z"] = (
    0.5 * df["z_returns_5yr"]
    + 0.3 * df["z_sharpe"]
    - 0.2 * df["z_std_dev"]
)


In [42]:
y = df["target_score_z"]
X = df[num_features + cat_features]


In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)


In [44]:
pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['standard_deviation',
                                                   'sharpe', 'sortino_ratio',
                                                   'expense_ratio',
                                                   'fund_size_cr',
                                                   'consistency_score',
                                                   'cost_efficiency_score',
                                                   'fund_stability', 'alpha',
                                                   'beta']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['category',
                                                   'risk_bucket'])])),
                ('model',
                 RandomForestRegressor(n_estimators=200, random_state=42))])

In [46]:
# Enforce numeric types globally
df[num_features] = (
    df[num_features]
    .replace("-", pd.NA)
    .apply(pd.to_numeric, errors="coerce")
)

# Features & target
X = df[num_features + cat_features]
y = df["target_score_raw"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Predict
y_pred = pipeline.predict(X_test)


In [47]:
y_pred = pipeline.predict(X_test)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

mae, rmse, r2


(5.786868813520148, np.float64(5.947862406172725), -10.653735607245881)

In [48]:
from scipy.stats import spearmanr

spearman_corr, _ = spearmanr(y_test, y_pred)
spearman_corr


np.float64(0.7941253478145283)

In [49]:
df["predicted_z_score"] = pipeline.predict(X)

top_actual = (
    df.sort_values("target_score_z", ascending=False)
      .head(5)["scheme_name"]
)

top_predicted = (
    df.sort_values("predicted_z_score", ascending=False)
      .head(5)["scheme_name"]
)

top_actual, top_predicted, set(top_actual).intersection(set(top_predicted))


(38       Quant Infrastructure Fund - Regular Plan IDCW
 15     Quant Infrastructure Fund - Regular Plan Growth
 17     Quant Infrastructure Fund - Regular Plan Growth
 113    Quant Infrastructure Fund - Regular Plan Growth
 81     Quant Infrastructure Fund - Regular Plan Growth
 Name: scheme_name, dtype: object,
 38       Quant Infrastructure Fund - Regular Plan IDCW
 15     Quant Infrastructure Fund - Regular Plan Growth
 17     Quant Infrastructure Fund - Regular Plan Growth
 113    Quant Infrastructure Fund - Regular Plan Growth
 11     Quant Infrastructure Fund - Regular Plan Growth
 Name: scheme_name, dtype: object,
 {'Quant Infrastructure Fund - Regular Plan Growth',
  'Quant Infrastructure Fund - Regular Plan IDCW'})

In [50]:
df[[
    "z_returns_5yr",
    "z_sharpe",
    "z_std_dev",
    "target_score_z"
]].corr()


,z_returns_5yr,z_sharpe,z_std_dev,target_score_z
z_returns_5yr,1.000000,-0.000173,0.475516,0.776824
z_sharpe,-0.000173,1.000000,0.110363,0.533126
z_std_dev,0.475516,0.110363,1.000000,0.135981
target_score_z,0.776824,0.533126,0.135981,1.000000


In [51]:
df["final_score"] = (
    0.4 * df["predicted_z_score"]
    + 0.6 * df["composite_score"]
)


In [52]:
risk_map = {
    "Low": ["Low Risk", "Moderately Low"],
    "Moderate": ["Moderate", "Moderately Low"],
    "High": ["High Risk", "Moderate"]
}



In [53]:
def filter_by_risk(df, user_risk):
    allowed = risk_map[user_risk]
    return df[df["risk_bucket"].isin(allowed)]



In [54]:
df_risk = filter_by_risk(df, "Moderate")
len(df_risk)


202

In [55]:
def create_return_band(r):
    if r < 10:
        return "Low"
    elif r < 15:
        return "Moderate"
    else:
        return "High"


In [56]:
df["return_band"] = df["returns_5yr"].apply(create_return_band)


In [57]:
df["return_band"].value_counts()


return_band
High        1819
Moderate    1015
Low          197
Name: count, dtype: int64

In [58]:
return_bands = {
    "Conservative": (6, 9),
    "Moderate": (9, 13),
    "Aggressive": (13, 18)
}

In [59]:
def filter_by_return(df, band):
    low, high = return_bands[band]
    return df[(df["returns_5yr"] >= low) & (df["returns_5yr"] <= high)]

In [60]:
def recommend_top_funds(
    df,
    user_risk,
    return_band,
    top_n=5
):
    df_filtered = filter_by_risk(df, user_risk)
    df_filtered = filter_by_return(df_filtered, return_band)

    return (
        df_filtered
        .sort_values("final_score", ascending=False)
        .head(top_n)
        [["scheme_name", "final_score", "returns_5yr", "risk_bucket"]]
    )


In [61]:
def sip_future_value(monthly_sip, annual_rate, years):
    r = annual_rate / 100 / 12
    n = years * 12
    fv = monthly_sip * (((1 + r)**n - 1) / r) * (1 + r)
    return round(fv, 2)


In [62]:
def lumpsum_future_value(amount, annual_rate, years):
    r = annual_rate / 100
    fv = amount * ((1 + r) ** years)
    return round(fv, 2)


In [63]:
def attach_projection(
    df_reco,
    investment_type,
    amount,
    years
):
    results = []

    for _, row in df_reco.iterrows():
        rate = row["returns_5yr"]

        if investment_type == "SIP":
            final_value = sip_future_value(amount, rate, years)
        else:
            final_value = lumpsum_future_value(amount, rate, years)

        results.append({
            "scheme_name": row["scheme_name"],
            "expected_cagr": rate,
            "investment_years": years,
            "final_value": final_value
        })

    return pd.DataFrame(results)


In [64]:
top_funds = recommend_top_funds(
    df,
    user_risk="Moderate",
    return_band="Moderate",
    top_n=5
)

projection = attach_projection(
    top_funds,
    investment_type="SIP",
    amount=5000,
    years=10
)

projection


,scheme_name,expected_cagr,investment_years,final_value
0,ICICI Pru Asset Allocator Fund - Direct Plan IDCW,11.556466,10,1131497.22
1,ICICI Pru Asset Allocator Fund - Regular Plan ...,12.943212,10,1229191.85
2,ICICI Pru Asset Allocator Fund - Regular Plan ...,12.920197,10,1227489.93
3,ICICI Pru Asset Allocator Fund - Regular Plan ...,12.513966,10,1197908.40
4,ICICI Pru Asset Allocator Fund - Regular Plan ...,12.999172,10,1233341.74


In [65]:
df_risk = filter_by_risk(df, "Moderate")
print(len(df_risk))


202


In [66]:
df_return = df[df["return_band"] == "Moderate"]
print(len(df_return))


1015


In [67]:
df_both = df_risk[df_risk["return_band"] == "Moderate"]
print(len(df_both))


145


In [68]:
top_funds = recommend_top_funds(
    df,
    user_risk="Moderate",
    return_band="Moderate",
    top_n=5
)

top_funds


,scheme_name,final_score,returns_5yr,risk_bucket
897,ICICI Pru Asset Allocator Fund - Direct Plan IDCW,0.339335,11.556466,Moderately Low
747,ICICI Pru Asset Allocator Fund - Regular Plan ...,0.334424,12.943212,Moderately Low
606,ICICI Pru Asset Allocator Fund - Regular Plan ...,0.329996,12.920197,Moderately Low
1017,ICICI Pru Asset Allocator Fund - Regular Plan ...,0.322630,12.513966,Moderately Low
821,ICICI Pru Asset Allocator Fund - Regular Plan ...,0.319466,12.999172,Moderately Low


In [69]:
projection = attach_projection(
    top_funds,
    investment_type="SIP",
    amount=10000,
    years=10
)
